# Media Index Database

In [ ]:
import mediatools

In [ ]:
import pathlib
import pydantic_settings

class Settings(pydantic_settings.BaseSettings):
    mongo_uri: str = "mongodb://127.0.0.1:27017/?directConnection=true"
    db_name: str = "media_archive"
    mongo_index_target: pathlib.Path

settings = Settings()

In [ ]:
import pymongo

async with pymongo.AsyncMongoClient(settings.mongo_uri, serverSelectionTimeoutMS=2000) as client:
    media_index = await mediatools.MediaIndexDB.from_client(client[settings.db_name])

    mindex = await mediatools.MediaIndexDB.from_client(
        client = client[settings.db_name]
    )
    await mindex.update_directory_index(settings.mongo_index_target, verbose=True)
    await mindex.dirs.find_by_path(settings.mongo_index_target)


    print(await mindex.dirs.count(settings.mongo_index_target))
    print(await mindex.videos.count(settings.mongo_index_target))

    sub_dirs = await mindex.dirs.find_by_path_prefix(settings.mongo_index_target)
    for sdir in sub_dirs:
        print(sdir.path.name, [imf.path.name for imf in sdir.image_files.values()])